In [ ]:
import polars as pl

def rollup_alpha_pl(
    column_name: str,
    delimiter: str = "#"
) -> pl.Expr:
    """
    Creates a Polars expression to roll up unique values in a column,
    handling numeric formatting, nulls, and pre-existing delimiters.
    Uses only native Polars expressions for optimal performance.

    Args:
        column_name: The name of the column to roll up.
        delimiter: The delimiter to use for joining and splitting values (default "#").

    Returns:
        A Polars expression suitable for group_by.agg() or .over().
    """

    # --- Step 1: Handle Mixed Types and Nulls/NaNs ---
    col_str = pl.col(column_name).cast(pl.String, strict=False)

    # Identify if the original value was numeric
    original_numeric_mask = pl.col(column_name).cast(pl.Float64, strict=False).is_not_null()

    # --- Step 2: Format Numeric Values (Remove trailing zeros and decimal point) ---
    # Apply formatting to the string representation
    formatted_col = (
        pl.when(original_numeric_mask)
        .then(
            pl.col(column_name).cast(pl.Float64, strict=False).cast(pl.String)
            .str.replace(r'0+$', '', literal=False)
            .str.replace(r'\.$', '', literal=False)
        )
        .otherwise(col_str)
    )

    # --- Step 3: Split the formatted column by delimiter -> List[str] ---
    split_col = formatted_col.str.split(delimiter)

    # --- Step 4: Filter the list elements (before explode) using list.eval ---
    # This applies the filter condition to each element *inside* the list
    # pl.element() refers to the current element being evaluated within the list
    filtered_split_col = split_col.list.eval(
        pl.element().filter(
            pl.element().is_not_null() &
            (pl.element() != "") &
            (pl.element() != "NaN")
        )
    )

    # --- Step 5: Explode the *filtered* list ---
    exploded_list = filtered_split_col.list.explode()

    # --- Step 6: Get unique values, sort, and concatenate ---
    return (
        exploded_list
        .unique()
        .sort()
        .str.concat(delimiter=delimiter)
    )


# --- Example Usage ---
df = pl.DataFrame({
    'group': ['A', 'A', 'A', 'B', 'B', 'C', 'A', 'B', 'A', 'D'],
    'mixed_col': ['x', 3.0, 'y', 'z#A', 3.20, None, 'x', 'w', 'B#C', 5.00],
    'other_col': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
}, strict=False)

print("Original DataFrame:")
print(df)

rollup_hash_expr = rollup_alpha_pl('mixed_col', delimiter='#')

grouped_result_hash = df.group_by('group').agg(rollup_hash_expr.alias('rolled_up_mixed_hash'))
print("\n--- GroupBy Aggregation (Delimiter: #) ---")
print(grouped_result_hash)

df_with_window_rollup_hash = df.with_columns(
    rollup_hash_expr.over('group').alias('window_rolled_up_mixed_hash')
)
print("\n--- Over/Window Operation (Delimiter: #) ---")
print(df_with_window_rollup_hash)

rollup_pipe_expr = rollup_alpha_pl('mixed_col', delimiter='|')
grouped_result_pipe = df.group_by('group').agg(rollup_pipe_expr.alias('rolled_up_mixed_pipe'))
print("\n--- GroupBy Aggregation (Delimiter: |) ---")
print(grouped_result_pipe)

df_with_window_rollup_pipe = df.with_columns(
    rollup_pipe_expr.over('group').alias('window_rolled_up_mixed_pipe')
)
print("\n--- Over/Window Operation (Delimiter: |) ---")
print(df_with_window_rollup_pipe)

rollup_dash_expr = rollup_alpha_pl('mixed_col', delimiter='-')
grouped_result_dash = df.group_by('group').agg(rollup_dash_expr.alias('rolled_up_mixed_dash'))
print("\n--- GroupBy Aggregation (Delimiter: -) ---")
print(grouped_result_dash)